In [1]:
import nltk
import pandas as pd
df= pd.read_csv("../Data/SMSSpamCollection", sep="\t", header=None)
df.columns=["label", "message"]
df["label"] = df["label"].map({"ham": 0, "spam": 1})
df.head(10)

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [2]:
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Lowercasing
df["message"] = df["message"].str.lower()


# Remove punctuation and special characters but keep numbers
tokenizer = RegexpTokenizer(r'\w+') 
df["message"] = df["message"].apply(tokenizer.tokenize)

# Lemmatization
lemmatizer = WordNetLemmatizer()
df["message"] = df["message"].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

# Remove stop words
stop_words = set(stopwords.words('english'))
df["message"] = df["message"].apply(lambda x: [word for word in x if word not in stop_words])


df["message"] = df["message"].apply(lambda x: ' '.join(x))
df.head(10)

,label,message
0,0,go jurong point crazy available bugis n great ...
1,0,ok lar joking wif u oni
2,1,free entry 2 wkly comp win fa cup final tkts 2...
3,0,u dun say early hor u c already say
4,0,nah think go usf life around though
5,1,freemsg hey darling 3 week word back like fun ...
6,0,even brother like speak treat like aid patent
7,0,per request melle melle oru minnaminunginte nu...
8,1,winner valued network customer selected receiv...
9,1,mobile 11 month u r entitled update latest col...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3, max_df=0.8, sublinear_tf=True)
# fitting the vectorizer on the training data and transforming both training and test data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## LightGBM

In [4]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, accuracy_score, precision_score,f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

param_grid = {'n_estimators': [100, 200, 300], 'num_leaves': [15, 31, 63], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1]}
grid_search_lgb = GridSearchCV(LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1), param_grid, cv=5, verbose=1, scoring='accuracy')
grid_search_lgb.fit(X_train_tfidf, y_train)
best_estimator_lgb = grid_search_lgb.best_estimator_
y_pred_lgb = best_estimator_lgb.predict(X_test_tfidf)
lgb_accuracy = accuracy_score(y_test, y_pred_lgb)
lgb_precision = precision_score(y_test, y_pred_lgb)
lgb_recall = recall_score(y_test, y_pred_lgb)
lgb_f1 = f1_score(y_test, y_pred_lgb)
print(f"LightGBM Accuracy: {lgb_accuracy:.4f} | Precision: {lgb_precision:.4f} | Recall: {lgb_recall:.4f} | F1-Score: {lgb_f1:.4f}")
print(confusion_matrix(y_test, y_pred_lgb))

Fitting 5 folds for each of 81 candidates, totalling 405 fits
LightGBM Accuracy: 0.9722 | Precision: 0.8986 | Recall: 0.8926 | F1-Score: 0.8956
[[951  15]
 [ 16 133]]


## XGBoost

In [5]:
from xgboost import XGBClassifier
param_grid= {'n_estimators': [100, 200, 300],'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1]}
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
grid_search = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss',scale_pos_weight = scale_pos_weight), param_grid, cv=5, verbose=1, scoring='f1')
grid_search.fit(X_train_tfidf, y_train)
best_estimator = grid_search.best_estimator_
y_pred_xgb = best_estimator.predict(X_test_tfidf)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
print(f"XGBoost Accuracy: {xgb_accuracy:.4f} | Precision: {xgb_precision:.4f} | Recall: {xgb_recall:.4f} | F1-Score: {xgb_f1:.4f}")
print(confusion_matrix(y_test, y_pred_xgb))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
XGBoost Accuracy: 0.9767 | Precision: 0.9301 | Recall: 0.8926 | F1-Score: 0.9110
[[956  10]
 [ 16 133]]
